In [8]:
end = datetime.utcnow()
start = end - timedelta(days=30)
start_iso = start.strftime("%Y-%m-%dT%H:%M:%SZ")
end_iso = end.strftime("%Y-%m-%dT%H:%M:%SZ")
print("DATE_RANGE", start_iso, end_iso)

balance_resp = requests.get(
    f"{settings.PAYPAL_API_BASE}/v1/reporting/balances",
    headers=headers,
    params={"currency_code": "USD"},
    timeout=30,
)
print("BALANCE_STATUS", balance_resp.status_code)
print("BALANCE_CONTENT_TYPE", balance_resp.headers.get("content-type"))
if balance_resp.status_code == 200:
    bal = balance_resp.json()
    print("BALANCE_KEYS", sorted(bal.keys()))
    print("BALANCE_COUNT", len(bal.get("balances", [])))
    if bal.get("balances"):
        print("BALANCE_SAMPLE", bal["balances"][0])
else:
    print("BALANCE_ERROR", balance_resp.text[:500])

search_resp = requests.get(
    f"{settings.PAYPAL_API_BASE}/v1/reporting/transactions",
    headers=headers,
    params={"start_date": start_iso, "end_date": end_iso, "page_size": 100},
    timeout=30,
)
print("SEARCH_STATUS", search_resp.status_code)
print("SEARCH_CONTENT_TYPE", search_resp.headers.get("content-type"))
if search_resp.status_code == 200:
    tx = search_resp.json()
    print("SEARCH_KEYS", sorted(tx.keys()))
    details = tx.get("transaction_details", [])
    print("TX_COUNT", len(details))
    if details:
        item = details[0]
        print("FIRST_TX_ID", item.get("transaction_info", {}).get("transaction_id"))
        print("FIRST_TX_TYPE", item.get("transaction_info", {}).get("transaction_type"))
        print("FIRST_TX_STATUS", item.get("transaction_info", {}).get("transaction_status"))
        print("FIRST_PAYER_EMAIL", (item.get("payer_info", {}) or {}).get("email"))
        print("FIRST_AMOUNT", (item.get("transaction_info", {}).get("transaction_amount") or {}).get("value"))
        print("FIRST_CURRENCY", (item.get("transaction_info", {}).get("transaction_amount") or {}).get("currency"))
        print("FIRST_SOURCE_REFERENCE", item.get("source_reference"))
else:
    print("SEARCH_ERROR", search_resp.text[:500])


NameError: name 'datetime' is not defined

In [ ]:
from datetime import datetime, timedelta
import json
import requests
from app.config import settings

print("PAYPAL_MODE", settings.PAYPAL_MODE)
print("PAYPAL_API_BASE", settings.PAYPAL_API_BASE)
print("HAS_CLIENT_ID", bool(settings.PAYPAL_CLIENT_ID))
print("HAS_CLIENT_SECRET", bool(settings.PAYPAL_CLIENT_SECRET))

# Reuse the existing authenticated backend token.
auth_resp = requests.post(
    f"{settings.PAYPAL_API_BASE}/v1/oauth2/token",
    auth=(settings.PAYPAL_CLIENT_ID, settings.PAYPAL_CLIENT_SECRET),
    data={"grant_type": "client_credentials"},
    timeout=30,
)
print("OAUTH_STATUS", auth_resp.status_code)
print("OAUTH_JSON_KEYS", sorted(auth_resp.json().keys()))
print("OAUTH_HAS_ACCESS_TOKEN", "access_token" in auth_resp.json())

auth_data = auth_resp.json()
headers = {"Authorization": f"Bearer {auth_data['access_token']}", "Content-Type": "application/json"}
print("TOKEN_TYPE", auth_data.get("token_type"))
print("EXPIRES_IN", auth_data.get("expires_in"))


PAYPAL_MODE sandbox
PAYPAL_API_BASE https://api-m.sandbox.paypal.com
HAS_CLIENT_ID True
HAS_CLIENT_SECRET True
OAUTH_STATUS 200
OAUTH_JSON_KEYS ['access_token', 'app_id', 'expires_in', 'nonce', 'scope', 'token_type']
OAUTH_HAS_ACCESS_TOKEN True
TOKEN_TYPE Bearer
EXPIRES_IN 31079


In [ ]:
import os
import sys
from pathlib import Path

# Use the backend directory as the app root for imports.
backend_dir = Path(r"c:\Users\ELCOT\Pictures\Screenshots\kabf\paypal-fraud-detection\backend")
if str(backend_dir) not in sys.path:
    sys.path.insert(0, str(backend_dir))

os.chdir(backend_dir)
print(f"WORKDIR={backend_dir}")
print("PYTHON", sys.executable)

# Sandbox-only guard: never change to live or perform payment state changes.
print("SANDBOX_ONLY=true")


WORKDIR=c:\Users\ELCOT\Pictures\Screenshots\kabf\paypal-fraud-detection\backend
PYTHON C:\Users\ELCOT\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe
SANDBOX_ONLY=true


# PayPal Sandbox final live verification

This notebook executes the final live verification against the PayPal Sandbox for the fraud detection project.

Security constraints:
- Never print credentials or access tokens.
- Never fabricate balance or transaction data.
- Keep all operations in Sandbox only.
- Do not capture or refund any payment.
- Do not modify unrelated features.


In [ ]:
import subprocess
import time
import requests
import json

backend_dir = Path(r"c:\Users\ELCOT\Pictures\Screenshots\kabf\paypal-fraud-detection\backend")
proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=str(backend_dir),
    env={**os.environ, "PYTHONPATH": str(backend_dir)},
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
try:
    time.sleep(6)
    import_resp = requests.post(
        "http://127.0.0.1:8000/api/paypal/import",
        params={"start_date": start_iso, "end_date": end_iso},
        timeout=120,
    )
    print("IMPORT_STATUS", import_resp.status_code)
    print("IMPORT_BODY", import_resp.text[:1200])
    import_json = import_resp.json() if 'application/json' in import_resp.headers.get('content-type','') else {}
    print("IMPORT_COUNT", import_json.get('imported_count'))
    print("DUPLICATES_SKIPPED", import_json.get('duplicates_skipped'))
    print("ALERT_COUNT", len(import_json.get('alerts', [])))

    second_resp = requests.post(
        "http://127.0.0.1:8000/api/paypal/import",
        params={"start_date": start_iso, "end_date": end_iso},
        timeout=120,
    )
    print("SECOND_IMPORT_STATUS", second_resp.status_code)
    print("SECOND_IMPORT_BODY", second_resp.text[:1200])
    second_json = second_resp.json() if 'application/json' in second_resp.headers.get('content-type','') else {}
    print("SECOND_IMPORT_COUNT", second_json.get('imported_count'))
    print("SECOND_DUPLICATES_SKIPPED", second_json.get('duplicates_skipped'))

    list_resp = requests.get("http://127.0.0.1:8000/api/paypal/transactions", timeout=60)
    list_json = list_resp.json() if 'application/json' in list_resp.headers.get('content-type','') else {}
    print("TRANSACTION_LIST_STATUS", list_resp.status_code)
    print("TRANSACTION_LIST_COUNT", list_json.get('count'))
    if list_json.get('results'):
        txid = list_json['results'][0]['paypal_transaction_id']
        print("FIRST_STORED_TX_ID", txid)
        print("FIRST_STORED_SOURCE", list_json['results'][0].get('source_reference'))
        lookup_resp = requests.get(
            "http://127.0.0.1:8000/api/paypal/transactions/search",
            params={"transaction_id": txid},
            timeout=60,
        )
        print("LOOKUP_STATUS", lookup_resp.status_code)
        print("LOOKUP_BODY", lookup_resp.text[:1200])

    alerts_resp = requests.get("http://127.0.0.1:8000/api/paypal/alerts", timeout=60)
    print("ALERTS_STATUS", alerts_resp.status_code)
    print("ALERTS_BODY", alerts_resp.text[:1200])
finally:
    try:
        proc.terminate()
        proc.wait(timeout=15)
        print("SERVER_EXIT", proc.returncode)
    except Exception as exc:
        print("SERVER_STOP_ERROR", repr(exc))


IMPORT_STATUS 200
IMPORT_BODY {"status":"ok","environment":"sandbox","imported_count":0,"duplicates_skipped":1,"message":"Retrieved transactions were stored with unique PayPal transaction IDs.","alerts":[]}
IMPORT_COUNT 0
DUPLICATES_SKIPPED 1
ALERT_COUNT 0
SECOND_IMPORT_STATUS 200
SECOND_IMPORT_BODY {"status":"ok","environment":"sandbox","imported_count":0,"duplicates_skipped":1,"message":"Retrieved transactions were stored with unique PayPal transaction IDs.","alerts":[]}
SECOND_IMPORT_COUNT 0
SECOND_DUPLICATES_SKIPPED 1
TRANSACTION_LIST_STATUS 200
TRANSACTION_LIST_COUNT 1
FIRST_STORED_TX_ID 4FT21609W98730452
FIRST_STORED_SOURCE /v1/reporting/transactions
LOOKUP_STATUS 200
LOOKUP_BODY {"status":"ok","found":true,"message":"PayPal Sandbox transaction found.","transaction":{"paypal_transaction_id":"4FT21609W98730452","parent_transaction_id":null,"resource_type":"T1900","transaction_type":null,"payer_name":null,"payer_email":null,"amount":"5000.00","currency":null,"transaction_status":"S